# Auto-Runbook Isolate Endpoint


## Specify Endpoint
Define the target agent details for isolation.


In [1]:
# Import required libraries
import requests
import argparse
import sys
import os
import subprocess
import json
from datetime import datetime

# Set target agent details
agent_name = "wazuh-agent-linux-1767224455"

print(f"Targeting Agent: {agent_name} ")


Targeting Agent: wazuh-agent-linux-1767224455 


## Validate Endpoint Connection
Check if the target agent is currently active and reachable by the Wazuh manager.


In [2]:
def check_agent_status(agent_name):
    """Check if agent is active using agent_control -l"""
    try:
        # List all agents
        result = subprocess.run(["sudo", "/var/ossec/bin/agent_control", "-l"], capture_output=True, text=True)
        
        # Search for the agent name in the output
        # Example line: "   ID: 004, Name: wazuh-agent-linux-1767224455, IP: any, Active"
        lines = result.stdout.split('\n')
        for line in lines:
            if f"Name: {agent_name}" in line:
                if "Active" in line:
                    # Extract ID
                    # ID: 004, Name: ...
                    agent_id = line.split(',')[0].split(':')[1].strip()
                    print(f"✅ Agent {agent_name} (ID: {agent_id}) is ACTIVE")
                    return agent_id
                else:
                    print(f"  Agent {agent_name} found but is NOT active")
                    return None
        
        print(f" Agent {agent_name} not found in agent_control list")
        return None
    except Exception as e:
        print(f" Error checking agent status: {e}")
        return None

# Run validation and get agent_id
agent_id = check_agent_status(agent_name)
is_active = agent_id is not None


✅ Agent wazuh-agent-linux-1767224455 (ID: 004) is ACTIVE


## Authentication
Provide the Wazuh API token for authentication. You can obtain this by running the authentication command in the terminal.


In [3]:
# Obtain Wazuh API token
def get_wazuh_token(url, user, password):
    """Authenticate with Wazuh API and return token"""
    print(f" Authenticating with Wazuh Manager at {url}...")
    try:
        response = requests.post(
            f"{url}:55000/security/user/authenticate",
            auth=(user, password),
            verify=False
        )
        if response.status_code == 200:
            token = response.json().get('data', {}).get('token')
            print("✅ Authentication successful")
            return token
        else:
            print(f" Authentication failed: {response.status_code}")
            print(response.text)
            return None
    except Exception as e:
        print(f" Connection error: {e}")
        return None

# Wazuh Credentials (from autodr.py)
wazuh_url = "https://localhost"
wazuh_user = "wazuh-wui"
wazuh_pass = "NZ5k.8gLaS4gPqiQFs+ArZDAwsgkvrBJ"

# Get the token
wazuh_token = get_wazuh_token(wazuh_url, wazuh_user, wazuh_pass)


 Authenticating with Wazuh Manager at https://localhost...


/home/stanislavtoman/autodr/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Authentication successful


## Isolate Endpoint
Trigger the active response to isolate the endpoint from the network.


In [4]:
def execute_isolation(agent_id, agent_name, wazuh_url="https://localhost", wazuh_token=None):
    """Execute firewall-drop active response"""
    print(f"\n RUNBOOK: Isolate Endpoint")
    print(f"   Target: {agent_name} (ID: {agent_id})")
    print("="*70)

    if not wazuh_token or wazuh_token == "YOUR_TOKEN_HERE":
        print(" Error: Valid Wazuh API token is required")
        return False

    if not agent_id:
        print(" Error: Agent ID is required")
        return False

    print("  📡 Cutting network connection...")

    headers = {
        'Authorization': f'Bearer {wazuh_token}',
        'Content-Type': 'application/json'
    }

    active_response = {
        "command": "firewall-drop",
        "arguments": ["-", "null", "0", "0"]
    }

    try:
        response = requests.put(
            f"{wazuh_url}:55000/active-response?agents_list={agent_id}",
            headers=headers,
            json=active_response,
            verify=False
        )

        if response.status_code == 200:
            print(f"  ✅ Endpoint {agent_name} isolated successfully")
            return True
        else:
            print(f"   Failed to isolate endpoint: {response.status_code}")
            print(response.text)
            return False

    except Exception as e:
        print(f"   Error: {e}")
        return False

# Execute isolation if agent is active and token is provided
if is_active and 'wazuh_token' in globals() and wazuh_token != "YOUR_TOKEN_HERE":
    execute_isolation(agent_id, agent_name, wazuh_token=wazuh_token)
else:
    print("  To execute isolation, ensure the agent is active and a valid wazuh_token is set in the Authentication section.")



 RUNBOOK: Isolate Endpoint
   Target: wazuh-agent-linux-1767224455 (ID: 004)
  📡 Cutting network connection...
  ✅ Endpoint wazuh-agent-linux-1767224455 isolated successfully


/home/stanislavtoman/autodr/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Recycle and Release Endpoint
Restore network access to the previously isolated endpoint.


In [5]:
def execute_release(agent_id, agent_name, wazuh_url="https://localhost", wazuh_token=None):
    """Restore network access to the endpoint"""
    print(f"\n🔓 RUNBOOK: Release Endpoint")
    print(f"   Target: {agent_name} (ID: {agent_id})")
    print("="*70)

    if not wazuh_token or wazuh_token == "YOUR_TOKEN_HERE":
        print(" Error: Valid Wazuh API token is required")
        return False

    print("  📡 Restoring network connection...")

    headers = {
        'Authorization': f'Bearer {wazuh_token}',
        'Content-Type': 'application/json'
    }

    # Example: Triggering a hypothetical 'firewall-release' or similar
    active_response = {
        "command": "firewall-drop", 
        "arguments": ["-", "null", "0", "0"],
        "alert": {"rule": {"id": "100004"}} # Hypothetical rule for release
    }

    try:
        response = requests.put(
            f"{wazuh_url}:55000/active-response?agents_list={agent_id}",
            headers=headers,
            json=active_response,
            verify=False
        )

        if response.status_code == 200:
            print(f"  ✅ Endpoint {agent_name} released successfully")
            return True
        else:
            print(f"   Failed to release endpoint: {response.status_code}")
            print(response.text)
            return False

    except Exception as e:
        print(f"   Error: {e}")
        return False

# Execute release if agent is active and token is provided
if is_active and 'wazuh_token' in globals() and wazuh_token != "YOUR_TOKEN_HERE":
    execute_release(agent_id, agent_name, wazuh_token=wazuh_token)
else:
    print("  To execute release, ensure the agent is active and a valid wazuh_token is set in the Authentication section.")



🔓 RUNBOOK: Release Endpoint
   Target: wazuh-agent-linux-1767224455 (ID: 004)
  📡 Restoring network connection...
  ✅ Endpoint wazuh-agent-linux-1767224455 released successfully


/home/stanislavtoman/autodr/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
